# Bagging Ensembles (Bootstrap Aggregating)

**Bagging**, short for **Bootstrap Aggregating**, is a powerful ensemble learning technique designed to improve the stability, accuracy, and generalization of machine learning algorithms. It belongs to the class of **parallel ensemble methods** and is primarily used to address the classic bias-variance tradeoff by significantly **reducing model variance** (overfitting).

In these detailed notes, we will cover:
1. **Core Concept:** Bootstrapping and Aggregating.
2. **The Bias-Variance Tradeoff:** Why Bagging reduces variance.
3. **The Four Types of Bagging:** Bagging, Pasting, Random Subspaces, and Random Patches.
4. **Out-of-Bag (OOB) Evaluation:** Concept, mathematical derivation, and validation benefits.
5. **Bagging Regressors:** Mechanics of ensembling continuous outputs.
6. **Best Practices:** Recommended hyperparameter ranges.
7. **Practical Python Demos:**
   - **Demo A:** Decision boundary visualization of the 4 bagging types vs. a single Decision Tree.
   - **Demo B:** `BaggingClassifier` with OOB score validation and hyperparameter tuning using `GridSearchCV`.
   - **Demo C:** `BaggingRegressor` vs. a single Decision Tree Regressor, highlighting prediction smoothing on non-linear data.
   - **Summary Checklist** for Bagging Ensembles.

## 1. What is Bagging (Bootstrap Aggregating)?

Bagging combines two core concepts to enhance predictive performance:

### A. Bootstrapping
Bootstrapping is a statistical resampling technique. Given a dataset of size $n$, we create multiple subsets (bootstrap samples) of size $n$ by **randomly sampling the rows with replacement**. 
*   Because sampling is done with replacement, some data points will be repeated multiple times in a single subset, while others will not be selected at all.
*   A separate base model (e.g., a Decision Tree) is trained independently and in parallel on each bootstrap sample.

### B. Aggregating
Once all base estimators are trained, we collect their individual predictions and aggregate them to form the final prediction:
*   **Classification:** Aggregated using **Majority Voting** (the mode of all class predictions).
*   **Regression:** Aggregated using the **Mean** (the average of all numerical predictions).

### Core Intuition
By training different models on different subsets of the data, we introduce **diversity** into the ensemble. Since each model has seen a slightly different slice of reality, their individual errors are independent. When we average their decisions, these independent errors cancel out, leaving a highly accurate collective prediction.

## 2. Why Use Bagging: Bias-Variance Tradeoff

### The Variance Problem
Many machine learning algorithms (especially fully grown, unconstrained Decision Trees) are **high variance, low bias** models. They are highly flexible and learn the training data (including noise and outliers) too well, resulting in overfitting. When tested on unseen data, their performance drops.

### How Bagging Solves It
Bagging is specifically designed to **reduce variance without increasing bias**:
*   Since each base estimator is trained on a different bootstrap sample, each model overfits to a slightly different subset of noise.
*   Aggregating (averaging) the individual estimators smooths out these noisy fluctuations.
*   The resulting ensemble has a **much smoother decision boundary** (in classification) or a **smoother curve** (in regression) than any single base model, leading to superior generalization.

$$\text{Var}(\bar{X}) = \frac{\sigma^2}{M}$$

*(Where $M$ is the number of independent estimators and $\sigma^2$ is the variance of a single estimator. Although estimators in bagging are not fully independent, this formula shows why averaging reduces variance.)*

## 3. The Four Types of Bagging

Depending on how the training subsets are constructed, bagging algorithms can be classified into four distinct types:

| Method | Row Sampling (Instances) | Column Sampling (Features) | Description |
| :--- | :---: | :---: | :--- |
| **Bagging** | Random with replacement | No (uses all features) | The standard approach. Combines bootstrap samples of rows with all features. |
| **Pasting** | Random *without* replacement | No (uses all features) | Base models are trained on random subsets of rows. Good for large datasets. |
| **Random Subspaces** | No (uses all rows) | Yes (with or without replacement) | Trains models on random subsets of features. Useful for high-dimensional data (e.g., image/text classification). |
| **Random Patches** | Random with replacement | Yes (with or without replacement) | Combines both row sampling and feature sampling. Best for very large, high-dimensional datasets. |

## 4. Out-of-Bag (OOB) Evaluation

### The Concept
When sampling rows with replacement to create bootstrap samples, some training rows are selected multiple times, while others are left out. The training instances that are **never selected** for a specific base estimator are called **Out-of-Bag (OOB)** samples for that estimator.

### Mathematical Derivation of OOB Proportion
For a dataset of size $n$, the probability of selecting a specific instance in a single draw is $\frac{1}{n}$. The probability of **not** selecting it is:

$$1 - \frac{1}{n}$$

Since a bootstrap sample consists of $n$ draws (with replacement), the probability that a specific instance is never selected in all $n$ draws is:

$$p = \left(1 - \frac{1}{n}\right)^n$$

Taking the mathematical limit as $n$ approaches infinity ($n \to \infty$):

$$\lim_{n \to \infty} \left(1 - \frac{1}{n}\right)^n = \frac{1}{e} \approx 0.36787 \approx 36.8\%$$

**Result:** For any reasonably large dataset, approximately **$37\%$ of the training instances** are omitted from each bootstrap sample. These are the OOB samples.

### Utility of OOB Score
Since the OOB samples were never seen by a base estimator during training, they can serve as a **built-in validation set**:
1.  For each training sample, find all base models that did *not* see it during training.
2.  Aggregate the predictions of these specific models on the sample.
3.  Compute the overall accuracy (OOB score) across all training samples.

This allows us to evaluate the ensemble's performance during training **without** needing a separate validation or test set.

## 5. Bagging Regressors

A **Bagging Regressor** works identically to a classifier, except for how predictions are aggregated.

### Aggregation Mechanism
Instead of majority voting, it calculates the **mean** of the outputs from the individual base models:

$$\hat{y}_{\text{ensemble}}(X) = \frac{1}{M} \sum_{i=1}^{M} R_i(X)$$

*(Where $R_i(X)$ is the prediction of the $i$-th regressor.)*

### Predictions Smoothing
Since complex regressors (like unconstrained Decision Tree Regressors) create jagged, highly sensitive predictions to capture every outlier, aggregating their outputs averages out these step-like sudden changes. This creates a **smooth, continuous prediction curve** that fits the underlying data distribution far better.

## 6. Best Practices & Key Hyperparameters

*   **`estimator`:** The base model to clone. Defaults to a Decision Tree, but can be an SVM, KNN, or Linear Model.
*   **`n_estimators`:** The number of base models to train. A good starting point is **$100$ to $500$**. Adding more estimators does not cause overfitting; it simply stabilizes performance (though it increases computation time).
*   **`max_samples`:** The proportion/number of rows to sample from the dataset. Recommended starting values are **$0.25$ to $0.50$** (sampling $25\%$ to $50\%$ of the data per model increases diversity).
*   **`max_features`:** The proportion/number of features to sample. Recommended to keep at $1.0$ (all features) unless working with very high-dimensional data, in which case values like $0.5$ are preferred.
*   **`bootstrap`:** Toggle between **Bagging** (`True`, with replacement) and **Pasting** (`False`, without replacement).
*   **`bootstrap_features`:** Whether features are drawn with replacement.
*   **`oob_score`:** Set to `True` to compute the Out-of-Bag score during training.
*   **`n_jobs`:** Set to `-1` to parallelize execution and speed up training by utilizing all CPU cores.

## 7. Code Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

# Set visualization styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 8. Demo A: Visualizing Decision Boundaries (The Four Bagging Types)

We will generate a synthetic 2D "moons" dataset to evaluate and visualize the decision boundaries of a single overfitted Decision Tree against the four bagging types:
1.  **Single Decision Tree** (No constraints).
2.  **Standard Bagging** (Row sampling with replacement, uses all features).
3.  **Pasting** (Row sampling without replacement, uses all features).
4.  **Random Subspaces** (No row sampling, column sampling with replacement).
5.  **Random Patches** (Both row sampling and column sampling).

In [ ]:
# 1. Generate 2D classification dataset
X, y = make_moons(n_samples=250, noise=0.35, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Instantiate estimators
dt_single = DecisionTreeClassifier(random_state=42)

bagging_std = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42), 
    n_estimators=100, max_samples=0.5, bootstrap=True, random_state=42
)

pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42), 
    n_estimators=100, max_samples=0.5, bootstrap=False, random_state=42
)

random_subspaces = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42), 
    n_estimators=100, max_samples=1.0, bootstrap=False, 
    max_features=0.5, bootstrap_features=True, random_state=42
)

random_patches = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42), 
    n_estimators=100, max_samples=0.5, bootstrap=True, 
    max_features=0.5, bootstrap_features=True, random_state=42
)

estimators = {
    'Single Decision Tree': dt_single,
    'Standard Bagging (Rows w/ Replacement)': bagging_std,
    'Pasting (Rows w/o Replacement)': pasting,
    'Random Subspaces (Features Only)': random_subspaces,
    'Random Patches (Rows & Features)': random_patches
}

for name, model in estimators.items():
    model.fit(X_train, y_train)

# Helper function to plot decision boundaries
def plot_boundary(model, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=30, label='Class 0')
    ax.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#e74c3c', edgecolor='k', s=30, label='Class 1')
    
    acc = accuracy_score(y_test, model.predict(X_test))
    ax.set_title(f"{title}\nTest Acc: {acc:.4f}", fontsize=10, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

# Plot all decision surfaces
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.ravel()

plot_boundary(dt_single, X_train, y_train, axes[0], 'Single Decision Tree')
plot_boundary(bagging_std, X_train, y_train, axes[1], 'Standard Bagging')
plot_boundary(pasting, X_train, y_train, axes[2], 'Pasting')
plot_boundary(random_subspaces, X_train, y_train, axes[3], 'Random Subspaces')
plot_boundary(random_patches, X_train, y_train, axes[4], 'Random Patches')

# Hide 6th subplot
axes[5].axis('off')
plt.tight_layout()
plt.show()

## 9. Demo B: OOB Score & Hyperparameter Tuning (Breast Cancer)

We will demonstrate the implementation of `BaggingClassifier` on the **Breast Cancer Dataset**. We will compute the Out-of-Bag (OOB) score to validate the model during training, and then perform `GridSearchCV` to optimize the number of estimators and sampling ratios.

In [ ]:
# 1. Load data & Split
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Train Bagging Classifier with OOB evaluation
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=500,
    max_samples=0.40,      # Use 40% of rows for each model
    bootstrap=True,
    oob_score=True,        # Enable out-of-bag scoring!
    n_jobs=-1,
    random_state=42
)

bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

print("OOB Evaluation results:")
print("="*55)
print(f"OOB Score (Validation Accuracy): {bag_clf.oob_score_:.4f}")
print(f"Test Set Accuracy:               {accuracy_score(y_test, y_pred):.4f}")
print("="*55)
print()

# 3. Inspect internal parameters (Estimator samples)
# Let's print the indices of the training samples used to train the first estimator
print(f"Total training samples: {len(X_train)}")
print(f"First estimator indices sample size: {len(bag_clf.estimator_samples_[0])}")
print(f"Sample indices used in Estimator 0 (first 15): \n{bag_clf.estimator_samples_[0][:15]}")
print()

# 4. Tuning Hyperparameters with GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200, 500],
    'max_samples': [0.25, 0.5, 0.75, 1.0],
    'bootstrap': [True, False]
}

grid = GridSearchCV(
    estimator=BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)
grid.fit(X_train, y_train)

print("GridSearchCV Tuning Results:")
print("-"*35)
print(f"Best Hyperparameters: {grid.best_params_}")
print(f"Best CV Score:        {grid.best_score_:.4f}")
print(f"Tuned Test Accuracy:  {accuracy_score(y_test, grid.predict(X_test)):.4f}")

## 10. Demo C: Bagging Regressor (Smoothing Predictions)

We will generate a synthetic non-linear 1D dataset (a noisy sine wave) and show how a single unconstrained `DecisionTreeRegressor` overfits the data points (chasing noise), while a `BaggingRegressor` averages the predictions to build a smooth, robust model.

In [ ]:
# 1. Generate noisy 1D sine wave data
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(100, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.2, X_reg.shape[0])

# 2. Train Single Tree Regressor (Low bias, high variance)
dt_reg = DecisionTreeRegressor(random_state=42)  # No depth restriction
dt_reg.fit(X_reg, y_reg)

# 3. Train Bagging Regressor using Decision Trees
bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=200,
    max_samples=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
bag_reg.fit(X_reg, y_reg)

# 4. Generate predictions on a grid
X_grid = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_pred_dt = dt_reg.predict(X_grid)
y_pred_bag = bag_reg.predict(X_grid)

# 5. Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Subplot 1: Single Decision Tree Regressor
axes[0].scatter(X_reg, y_reg, color='darkorange', edgecolor='black', s=45, label='Actual Data')
axes[0].plot(X_grid, y_pred_dt, color='red', linewidth=2.5, label='Single Tree Predictor')
axes[0].set_title(f"Single Decision Tree Regressor\nR²: {r2_score(y_reg, dt_reg.predict(X_reg)):.4f} (Overfitting)", fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature X')
axes[0].set_ylabel('Target y')
axes[0].legend(frameon=True, facecolor='white')

# Subplot 2: Bagging Regressor
axes[1].scatter(X_reg, y_reg, color='darkorange', edgecolor='black', s=45, label='Actual Data')
axes[1].plot(X_grid, y_pred_bag, color='purple', linewidth=3.0, label='Bagged Trees Predictor')
axes[1].set_title(f"Bagging Regressor (200 Trees)\nR²: {r2_score(y_reg, bag_reg.predict(X_reg)):.4f} (Robust Fit)", fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature X')
axes[1].set_ylabel('Target y')
axes[1].legend(frameon=True, facecolor='white')

plt.tight_layout()
plt.show()

# 6. Metrics
print("Performance Metrics:")
print("-"*30)
print(f"Single Decision Tree MSE: {mean_squared_error(y_reg, dt_reg.predict(X_reg)):.4f}")
print(f"Bagging Regressor MSE:    {mean_squared_error(y_reg, bag_reg.predict(X_reg)):.4f}")

## Summary Checklist for Bagging Ensembles

1.  **Core Intuition:** Bagging = **Bootstrapping** (sampling rows with replacement) + **Aggregating** (majority vote for classification, mean for regression).
2.  **Variance Reduction:** Specifically targets **high-variance, low-bias** base learners (like unconstrained decision trees) to reduce overfitting without increasing bias.
3.  **Identify the Four Bagging Types:**
    *   *Bagging:* Row sampling with replacement, uses all features.
    *   *Pasting:* Row sampling without replacement, uses all features.
    *   *Random Subspaces:* All rows, feature sampling.
    *   *Random Patches:* Both row sampling and feature sampling.
4.  **Understand the Out-of-Bag (OOB) Math:** The probability that a training row is never selected in a bootstrap sample converges to $1/e \approx 37\%$. These act as a built-in validation set.
5.  **Utilize OOB Scores:** Set `oob_score=True` to get a validation accuracy metric directly during training, eliminating the need for a separate train-validation split.
6.  **Inspection:** Use `model.estimator_samples_` to retrieve the sample indices assigned to each individual base estimator.
7.  **Speed Up Training:** Set `n_jobs=-1` to distribute training of the parallel estimators across all available CPU cores.
8.  **Smoothing Continuous Target Predictions:** In `BaggingRegressor`, averaging predictions smooths out the step-like boundaries of single regressors, creating stable predictions.
9.  **Tuning Guidelines:** Set `max_samples` between $0.25$ and $0.50$ to maximize model diversity, and use `n_estimators=100` to `500` for stable results.